## Mixture of Agents
Here we can use the Python SDK to develop a mixture of agents, then save the agent to a config.yaml and run it from there.

In [1]:
import os
import sys

# Import the NeMo-Agent-Toolkit module
module_path = os.path.abspath('../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)

In [2]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

In [3]:
system_prompt = """
Answer the following questions as best you can. You may communicate and collaborate with various experts to answer the
questions:

{tools}

You may respond in one of two formats.
Use the following format exactly to communicate with an expert:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action (if there is no required input, include "Action Input: None")
Observation: wait for the expert to respond, do not assume the expert's response

... (this Thought/Action/Action Input/Observation can repeat N times.)
Use the following format once you have the final answer:

Thought: I now know the final answer
Final Answer: the final answer to the original input question
"""

In [ ]:
from nat.agent.sdk import NatReActAgent
from nat.agent.sdk import ToolCallingAgent
from nat.llm.sdk import NimLLM
from nat.plugins.langchain.sdk import CodeGenerationTool
from nat.plugins.langchain.sdk import WikiSearchTool
from nat.tool.sdk import CurrentTimeTool
from nat.utils.sdk.nat_workflow import NatWorkflow
from nat_simple_calculator.sdk import CalculatorToolGroup

agent_orchestrator_llm = NimLLM(
    model_name="nvdev/meta/llama-3.1-405b-instruct",
    temperature=0.2,
    max_tokens=250,
    name="agent_orchestrator",
)
agent_executor_llm = NimLLM(
    model_name="nvdev/meta/llama-3.3-70b-instruct",
    temperature=0,
    max_tokens=250,
    name="agent_executor",
)
calculator_tool_group = CalculatorToolGroup(
    name="calculator",
)

# Define a list of tools
wikipedia_search_tool = WikiSearchTool(
    max_results=3,
    name="wiki_search",
)
current_time_tool = CurrentTimeTool(
    name="current_datetime",
)
generate_code_tool = CodeGenerationTool(
    programming_language="Python",
    description="Useful to generate Python code. For any questions about code generation, you must only use this tool!",
    llm=agent_orchestrator_llm,
    verbose=True,
    name="code_generation",
)

# Define a list of agents (agents can be used as tools)
math_agent = ToolCallingAgent(
    tools=[calculator_tool_group],
    llm=agent_executor_llm,
    verbose=True,
    handle_tool_errors=True,
    description="Useful for performing simple mathematical calculations.",
    name="math_agent",
)
internet_agent = ToolCallingAgent(
    tools=[wikipedia_search_tool, current_time_tool],
    llm=agent_executor_llm,
    verbose=True,
    handle_tool_errors=True,
    description="Useful for performing simple internet searches.",
    name="internet_agent",
)

# The orchestrator agent can use other agents as tools
agent = NatReActAgent(
    tools=[math_agent, internet_agent, generate_code_tool],
    llm=agent_orchestrator_llm,
    verbose=True,
    parse_agent_response_max_retries=2,
    system_prompt=system_prompt,
)

nat_workflow = NatWorkflow(
    entrypoint=agent,
)

/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import os
from pathlib import Path

path_to_yaml = Path(os.getcwd(), "config", "config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the workflow to a config file
nat_workflow.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


functions:
  math_agent:
    _type: tool_calling_agent
    llm_name: agent_executor
    verbose: true
    description: Useful for performing simple mathematical calculations.
    tool_names:
    - calculator
    handle_tool_errors: true
  internet_agent:
    _type: tool_calling_agent
    llm_name: agent_executor
    verbose: true
    description: Useful for performing simple internet searches.
    tool_names:
    - wiki_search
    - current_datetime
    handle_tool_errors: true
  wiki_search:
    _type: wiki_search
    max_results: 3
  current_datetime:
    _type: current_datetime
  code_generation:
    _type: code_generation
    llm_name: agent_orchestrator
    verbose: true
    programming_language: Python
    description: |-
      Useful to generate Python code. For any questions about code generation, you must only use this
      tool!

function_groups:
  calculator:
    _type: calculator

llms:
  agent_orchestrator:
    _type: nim
    model: nvdev/meta/llama-3.1-405b-instruct
    

In [ ]:
await nat_workflow.prompt("Who was Djikstra?")

/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:715: UserWarning: Model 'nvdev/meta/llama-3.3-70b-instruct' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:715: UserWarning: Model 'nvdev/meta/llama-3.3-70b-instruct' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
Discovered frameworks: {<LLMFrameworkEnum.LANGCHAIN: 'langchain'>} in function code_generation_tool by inspecting source. It is recommended and more reliable to instead add the used LLMFrameworkEnum types in the framework_wrappers argument when calling @register_function.


"Edsger W. Dijkstra was a computer scientist who developed Dijkstra's algorithm, a well-known algorithm for finding the shortest path between nodes in a weighted graph."

In [ ]:
from pathlib import Path

from nat.eval.sdk import RagasEvaluator
from nat.utils.sdk.nat_evaluation import EvalDatasetJsonConfig
from nat.utils.sdk.nat_evaluation import NatEvaluation

path_to_dataset = Path(os.path.curdir, "../../../../", "examples/agents/data/wikipedia.json").resolve()

accuracy_evaluator = RagasEvaluator(
    llm=agent_orchestrator_llm,
    metric="AnswerAccuracy",
    name="accuracy"
)

relevance_evaluator = RagasEvaluator(
    llm=agent_orchestrator_llm,
    metric="ContextRelevance",
    name="relevance"
)

response_groundedness_evaluator = RagasEvaluator(
    llm=agent_orchestrator_llm,
    metric="ResponseGroundedness",
    name="groundedness"
)

evaluation = NatEvaluation(
    output_dir=Path(".tmp/nat/examples/mixture_of_agents/"),
    dataset=EvalDatasetJsonConfig(file_path=path_to_dataset),
    evaluators=[accuracy_evaluator, relevance_evaluator, response_groundedness_evaluator],
)

nat_workflow.add_evaluator(evaluation)


In [ ]:
path_to_yaml = Path(os.getcwd(), "config", "eval_config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the workflow to a config file
nat_workflow.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())


In [ ]:
await nat_workflow.evaluate()
